# TF-IDF & Word Embeding

# TF-IDF & Word Embeding

## Library

In [ ]:
!pip install plotly
!pip install --upgrade gensim

In [ ]:
from gensim.models import Word2Vec, FastText
import pandas as pd
import re

from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer

from matplotlib import pyplot as plt
import plotly.graph_objects as go

import numpy as np

import warnings
warnings.filterwarnings('ignore')

term frekuensi

In [ ]:
pta_df_wfq = pd.read_excel("pta_word_frequency.xlsx", engine='openpyxl')
pta_df_wfq.head(10)

,kata,jumlah
0,pengaruh,5560
1,kerja,5394
2,teliti,4385
3,variabel,3677
4,usaha,2538
5,signifikan,2494
6,uji,2370
7,karyawan,2273
8,nilai,1912
9,hasil,1788


cleaning abstrak

In [ ]:
df = pd.read_csv('pta_abstrak.csv')

In [ ]:
clean_txt = []
for w in range(len(df.abstrak_normal)):
    # ubah NaN jadi string kosong
    desc = str(df['abstrak_normal'][w]).lower()

    # remove punctuation
    desc = re.sub('[^a-zA-Z]', ' ', desc)

    # remove tags
    desc = re.sub("&lt;/?.*?&gt;", " ", desc)

    # remove digits and special chars
    desc = re.sub("(\\d|\\W)+", " ", desc)

    clean_txt.append(desc.strip())

df['clean'] = clean_txt
df.head()

,abstrak_normal,clean
0,abstrak aliyah pengaruh faktor latih kembang p...,abstrak aliyah pengaruh faktor latih kembang p...
1,tuju teliti persepsi band association langgan ...,tuju teliti persepsi band association langgan ...
2,NaN,nan
3,aplikasi nyata manfaat teknologi informasi kom...,aplikasi nyata manfaat teknologi informasi kom...
4,abstrak teliti metode kuantitatif tekan uji hi...,abstrak teliti metode kuantitatif tekan uji hi...


## TF-IDF

In [ ]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['clean'])

# ubah ke DataFrame biar keliatan
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=vectorizer.get_feature_names_out()
)

In [ ]:
print("\nTF-IDF shape:", tfidf_df.shape)
print(tfidf_df.head())


TF-IDF shape: (1031, 5445)
    aa  aaaamanahsyariah   ab  abadi  abah  abai  abal  abas  abc  abdu  ...  \
0  0.0               0.0  0.0    0.0   0.0   0.0   0.0   0.0  0.0   0.0  ...   
1  0.0               0.0  0.0    0.0   0.0   0.0   0.0   0.0  0.0   0.0  ...   
2  0.0               0.0  0.0    0.0   0.0   0.0   0.0   0.0  0.0   0.0  ...   
3  0.0               0.0  0.0    0.0   0.0   0.0   0.0   0.0  0.0   0.0  ...   
4  0.0               0.0  0.0    0.0   0.0   0.0   0.0   0.0  0.0   0.0  ...   

   zakiyatus  zaman  zan  zavgren  zebra  zmiewski  zmijewski  \
0        0.0    0.0  0.0      0.0    0.0       0.0        0.0   
1        0.0    0.0  0.0      0.0    0.0       0.0        0.0   
2        0.0    0.0  0.0      0.0    0.0       0.0        0.0   
3        0.0    0.0  0.0      0.0    0.0       0.0        0.0   
4        0.0    0.0  0.0      0.0    0.0       0.0        0.0   

   zmijewskiterhadap  zulkifli  zulkiflimsi  
0                0.0       0.0          0.0  
1       

## Word Embeding

In [ ]:
corpus = []
for col in df.clean:
   word_list = col.split(" ")
   corpus.append(word_list)

#show first value
corpus[0:1]

#generate vectors from corpus
model = Word2Vec(corpus, min_count=1, vector_size = 56)

In [ ]:
# explore embeddings using cosine similarity
print("Kata mirip dengan 'penelitian':")
print(model.wv.most_similar('penelitian', topn=5))

print("\nKata mirip dengan 'data':")
print(model.wv.most_similar('data', topn=5))

# contoh cosmul
print("\nCosmul (penelitian + sistem - data):")
print(model.wv.most_similar_cosmul(positive=['penelitian', 'sistem'], negative=['data'], topn=5))

# doesnt_match: cari kata yang tidak sesuai konteks
print("\nKata yang tidak cocok dalam ['penelitian', 'data', 'sistem', 'informasi']:")
print(model.wv.doesnt_match("penelitian data sistem informasi".split()))

# save embeddings
filename = 'pta_embeddings.txt'
model.wv.save_word2vec_format(filename, binary=False)
print(f"\nEmbeddings disimpan ke {filename}")


Kata mirip dengan 'penelitian':
[('landas', 0.9780809879302979), ('gambar', 0.9647425413131714), ('deskripsi', 0.9558202624320984), ('jadi', 0.9520087838172913), ('promosional', 0.9484675526618958)]

Kata mirip dengan 'data':
[('primer', 0.9833757877349854), ('kumpul', 0.9514186382293701), ('reduksi', 0.9456626772880554), ('olah', 0.9438228607177734), ('gun', 0.9437822699546814)]

Cosmul (penelitian + sistem - data):
[('psikologi', 1.3545849323272705), ('alu', 1.353352665901184), ('nyaman', 1.351493000984192), ('wicaksono', 1.3468221426010132), ('transparansi', 1.3453493118286133)]

Kata yang tidak cocok dalam ['penelitian', 'data', 'sistem', 'informasi']:
data

Embeddings disimpan ke pta_embeddings.txt


In [ ]:
class MyTokenizer:
    def fit_transform(self, texts):
        # Tokenisasi sederhana: lowercase + split
        return [str(text).lower().split() for text in texts]

class MeanEmbeddingVectorizer:
    def __init__(self, word2vec_model):
        self.word2vec = word2vec_model
        # Perbaikan: gunakan vector_size (Gensim ≥ 4.0)
        self.dim = word2vec_model.wv.vector_size

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_tokenized = MyTokenizer().fit_transform(X)
        embeddings = []
        for words in X_tokenized:
            # Ambil vektor hanya untuk kata yang ada di vocab
            valid_vectors = [
                self.word2vec.wv[word] for word in words
                if word in self.word2vec.wv
            ]
            if valid_vectors:
                embeddings.append(np.mean(valid_vectors, axis=0))
            else:
                embeddings.append(np.zeros(self.dim))
        return np.array(embeddings)

    def fit_transform(self, X, y=None):
        return self.transform(X)

In [ ]:
df.shape

(1031, 2)

In [ ]:
mean_embedding_vectorizer = MeanEmbeddingVectorizer(model)
mean_embedded = mean_embedding_vectorizer.fit_transform(df['clean'])

In [ ]:
df['array']=list(mean_embedded)

In [ ]:
df.head(5)

,abstrak_normal,clean,array
0,abstrak aliyah pengaruh faktor latih kembang p...,abstrak aliyah pengaruh faktor latih kembang p...,"[-0.72181696, 0.19165173, 0.5548718, -0.139131..."
1,tuju teliti persepsi band association langgan ...,tuju teliti persepsi band association langgan ...,"[-0.8840407, 0.8277604, 0.48036844, -0.5114786..."
2,NaN,nan,"[-0.014413638, 0.009600126, 0.0088578295, 0.00..."
3,aplikasi nyata manfaat teknologi informasi kom...,aplikasi nyata manfaat teknologi informasi kom...,"[-0.6114156, 0.40892786, 0.33070534, -0.426597..."
4,abstrak teliti metode kuantitatif tekan uji hi...,abstrak teliti metode kuantitatif tekan uji hi...,"[-0.7259138, 0.43529272, 0.5896102, -0.0041221..."


In [ ]:
df['embedding_length'] = df['array'].str.len()

In [ ]:
print(df['embedding_length'])

0       56
1       56
2       56
3       56
4       56
        ..
1026    56
1027    56
1028    56
1029    56
1030    56
Name: embedding_length, Length: 1031, dtype: int64


In [ ]:
df.shape

(1031, 4)

In [ ]:
num_features = len(df['array'].iloc[0])  # asumsi semua list punya panjang sama
columns = [f'f{i+1}' for i in range(num_features)]

# Inisialisasi dictionary untuk menampung data per kolom
data_dict = {col: [] for col in columns}

# Looping setiap baris di kolom 'embedding'
for embedding_list in df['array']:
    for i, value in enumerate(embedding_list):
        data_dict[f'f{i+1}'].append(value)

# Buat DataFrame dari dictionary
embedding_df = pd.DataFrame(data_dict)

print(embedding_df)

            f1        f2        f3        f4        f5        f6        f7  \
0    -0.721817  0.191652  0.554872 -0.139131  0.327110  0.211324  0.259094   
1    -0.884041  0.827760  0.480368 -0.511479  0.500783 -0.173992  0.337116   
2    -0.014414  0.009600  0.008858  0.006671 -0.007198  0.017463  0.004589   
3    -0.611416  0.408928  0.330705 -0.426597  0.417662  0.092553  0.333105   
4    -0.725914  0.435293  0.589610 -0.004122  0.506253  0.427021  0.511212   
...        ...       ...       ...       ...       ...       ...       ...   
1026 -0.521373  0.340408  0.091436 -0.078839  0.227025  0.061853  0.304985   
1027 -0.351435 -0.062910  0.424787  0.287792  0.394506  0.379045  0.572614   
1028 -0.672118  0.615127  0.386436 -0.130957  0.486488  0.237188  0.587592   
1029 -1.072890  0.533722  0.527428 -0.357205  0.803109  0.203280  0.551466   
1030 -0.481070  0.166730  0.562769 -0.092377  0.326598  0.245848  0.237702   

            f8        f9       f10  ...       f47       f48    

In [ ]:
embedding_df

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,...,f47,f48,f49,f50,f51,f52,f53,f54,f55,f56
0,-0.721817,0.191652,0.554872,-0.139131,0.327110,0.211324,0.259094,-0.677944,-0.164583,-0.762556,...,-0.269124,0.680719,0.690740,0.177619,-0.077400,0.518936,-0.168150,0.484092,0.059197,-0.664321
1,-0.884041,0.827760,0.480368,-0.511479,0.500783,-0.173992,0.337116,-0.506021,-0.075400,-0.625739,...,0.197702,0.039607,-0.070292,0.188167,0.149884,0.537672,0.175252,0.552733,0.181610,-0.036966
2,-0.014414,0.009600,0.008858,0.006671,-0.007198,0.017463,0.004589,-0.000090,0.001711,0.003228,...,-0.011677,-0.013732,-0.006864,-0.004532,0.004385,-0.003233,0.015214,-0.001837,-0.007297,0.005300
3,-0.611416,0.408928,0.330705,-0.426597,0.417662,0.092553,0.333105,-0.496440,-0.000448,-0.305153,...,-0.034931,0.145736,0.083755,0.217982,-0.043967,0.395572,0.030794,0.555230,0.190031,-0.246524
4,-0.725914,0.435293,0.589610,-0.004122,0.506253,0.427021,0.511212,-0.745717,0.016443,-1.346280,...,-0.204677,0.766619,0.627325,0.554609,-0.374875,0.483934,0.016794,0.575350,0.153184,-0.719583
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1026,-0.521373,0.340408,0.091436,-0.078839,0.227025,0.061853,0.304985,-0.540568,-0.190380,-0.670780,...,0.271170,0.378754,-0.152256,0.278351,-0.068713,0.328742,0.300022,0.445816,0.012950,-0.081562
1027,-0.351435,-0.062910,0.424787,0.287792,0.394506,0.379045,0.572614,-0.856786,-0.083686,-1.283797,...,-0.294884,1.193831,0.745955,0.555445,-0.473417,0.419891,0.080825,0.303082,0.126624,-1.028344
1028,-0.672118,0.615127,0.386436,-0.130957,0.486488,0.237188,0.587592,-0.611652,-0.016935,-1.174092,...,0.144634,0.468403,0.020847,0.533143,-0.268326,0.443104,0.348300,0.601595,0.182515,-0.314323
1029,-1.072890,0.533722,0.527428,-0.357205,0.803109,0.203280,0.551466,-0.549229,0.045890,-0.473108,...,-0.019885,0.161115,0.043312,0.398034,-0.242760,0.338560,0.019562,0.825966,0.412568,-0.226020


In [ ]:
embedding_df.shape

(1031, 56)